In [8]:
from typing import List

from agent.components.GaussianProcess import GASK
from notebooks.summersoc.globals import PATH_METRICS_DEMO_EXPLORE, PATH_MODEL_LIST, SPLIT_DATA_INTO_X_PARTS, \
    ITERATE_THROUGH_X_PARTS, DIR_CANDIDATES, PATH_METRICS_DEMO_EXPLOIT, RUNS_PER_CONFIG
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import joblib

plt.style.use('default')

from agent.components import RASK
from agent.components.commons import ServiceType, theoretical_param_bounds, ServiceVar
from agent.components.commons import SloSet

services = [ServiceType.QR, ServiceType.CV, ServiceType.PC]
slos = [SloSet.DEFAULT, SloSet.HIGH_PERF, SloSet.LOW_COST, SloSet.HIGH_QUALITY]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
_df_explore = pd.read_csv(PATH_METRICS_DEMO_EXPLORE)
_df_exploit = pd.read_csv(PATH_METRICS_DEMO_EXPLOIT)
df_joint = pd.concat([_df_explore, _df_exploit])
df_preprocessed = RASK.preprocess_data(df_joint)

gp_list = joblib.load(PATH_MODEL_LIST)['gp_list']
rm_list = joblib.load(PATH_MODEL_LIST)['rm_list']

INFO:multiscale:Training data contains service types <StringArray>
[  'elastic-workbench-qr-detector',   'elastic-workbench-cv-analyzer',
 'elastic-workbench-pc-visualizer']
Length: 3, dtype: str


## **Plan**: Intent-based Inference

Find optimal parameter configs for each SLO x Service combination after seeing certain data shares

**TODO**: This here could actually be the place where we introduce out multi-dimensional methodology

![Monitoring Infrastructure](img/RASK_Methodology.jpg)

### Infer parameter assignments

#### Variant 1: Using deterministic gradient descent

In [3]:
# TODO: I need to extract the predictions using the SLORegistry_v1


from agent.components.Optimizer_v2 import run_optimizer_multi


def get_all_inferred_assignments(model_list: List[GASK] | List[RASK.RASK]):
    all_inferred_assignments = []

    for i in range(ITERATE_THROUGH_X_PARTS):
        for q in slos:
            for s in services:
                data_ratio = (i + 1) / ITERATE_THROUGH_X_PARTS
                if type(model_list[0]) is RASK.RASK:
                    model: RASK.RASK = model_list[i]
                else:
                    model: GASK = model_list[i][s]

                # Run optimizer to find the best configuration

                solutions = run_optimizer_multi(s, q.value, model, theoretical_param_bounds[s], runs=5)
                fitness, config = max(solutions, key=lambda x: x[0])

                # Predict performance (mu, sigma) for the chosen configuration
                x_state = {ServiceVar.COST: config[0], ServiceVar.QUALITY: config[1]}
                x_state = x_state | ({ServiceVar.MODEL: config[2]} if s == ServiceType.CV else {})
                pred = model.predict(s, "max_tp", x_state)  # this gives a distribution for GP

                # Store everything needed for the next block
                # We include empirical_var_bounds here as it changes per iteration
                all_inferred_assignments.append({
                    'data_rate': data_ratio,
                    'service_type': s.value,
                    'p_fitness': fitness,
                    'dist': pred,
                    'cores': x_state[ServiceVar.COST],
                    'data_quality': x_state[ServiceVar.QUALITY],
                    'model_size': x_state[ServiceVar.MODEL] if s == ServiceType.CV else -1,
                })

                print(f"Predicted fitness for {s.value} and {q.name} with {data_ratio * 100}% training data: {fitness}")
    return all_inferred_assignments


In [4]:
rm_assignments = get_all_inferred_assignments(rm_list)

Predicted fitness for elastic-workbench-qr-detector and DEFAULT with 1.6666666666666667% training data: 0.67
Predicted fitness for elastic-workbench-cv-analyzer and DEFAULT with 1.6666666666666667% training data: 0.8958031409103726
Predicted fitness for elastic-workbench-pc-visualizer and DEFAULT with 1.6666666666666667% training data: 0.8958031409103726
Predicted fitness for elastic-workbench-qr-detector and HIGH_PERF with 1.6666666666666667% training data: 0.9500000000000001
Predicted fitness for elastic-workbench-cv-analyzer and HIGH_PERF with 1.6666666666666667% training data: 0.9500000000000001
Predicted fitness for elastic-workbench-pc-visualizer and HIGH_PERF with 1.6666666666666667% training data: 0.9500000000000001
Predicted fitness for elastic-workbench-qr-detector and LOW_COST with 1.6666666666666667% training data: 0.7158267479373799
Predicted fitness for elastic-workbench-cv-analyzer and LOW_COST with 1.6666666666666667% training data: 0.7158267479373799
Predicted fitness 

#### Variant 2: Using stochastic gradient descent

In [5]:
# gp_assignments = get_all_inferred_assignments(gp_list)

#### Export to candidate solution script, with each config repeated x times

In [10]:

candidate_repetitions = []

# Iterate through the list in steps of 3 (the size of your service triple)
for i in range(0, len(rm_assignments), 3):
    # Extract the current triple of rows
    triple = rm_assignments[i: i + 3]

    # Repeat this specific triple for the number of runs
    for run_idx in range(runs_per_config):
        for row in triple:
            new_row = row.copy()
            new_row['rep'] = run_idx + 1
            candidate_repetitions.append(new_row)

file_name = f'candidate_solutions_{ITERATE_THROUGH_X_PARTS}_{SPLIT_DATA_INTO_X_PARTS}_{RUNS_PER_CONFIG}.csv'
df_candidates = pd.DataFrame(candidate_repetitions)
df_candidates.to_csv(DIR_CANDIDATES / file_name, index=False)
